In [4]:
from pathlib import Path
import json
import os
import time
from dotenv import load_dotenv
from datetime import datetime, timezone


def find_project_root():
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if (path / "prompts" / "system_v0.md").exists():
            return path

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto TI-support-RAG."
    )


PROJECT_ROOT = find_project_root()

PROMPT_PATH = PROJECT_ROOT / "prompts" / "system_v0.md"
TEST_PATH = PROJECT_ROOT / "test" / "test_cases.json"
RESULTS_PATH = PROJECT_ROOT / "docs" / "results.json"

MODEL = "openai/gpt-oss-20b"
PROMPT_VERSION = "system_v0"

print(f"Proyecto:   {PROJECT_ROOT}")
print(f"Prompt:     {PROMPT_PATH}")
print(f"Casos:      {TEST_PATH}")
print(f"Resultados: {RESULTS_PATH}")
print(f"Modelo:     {MODEL}")
print(f"Versión:    {PROMPT_VERSION}")

Proyecto:   /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG
Prompt:     /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/prompts/system_v0.md
Casos:      /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/test/test_cases.json
Resultados: /home/dvdm12/Documents/Obsidian Vault/TI-support-RAG/docs/results.json
Modelo:     openai/gpt-oss-20b
Versión:    system_v0


In [5]:
load_dotenv(PROJECT_ROOT / ".env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(
        "No se encontró GROQ_API_KEY. "
        "Verifica el archivo .env en la raíz del proyecto."
    )

print("GROQ_API_KEY configurada: True")

GROQ_API_KEY configurada: True


In [6]:
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

with open(TEST_PATH, "r", encoding="utf-8") as f:
    TEST_CASES = json.load(f)

print(f"Prompt cargado: {len(SYSTEM_PROMPT)} caracteres")
print(f"Casos de prueba cargados: {len(TEST_CASES)}")

for case in TEST_CASES:
    print(f"- {case['id']}: {case['tipo']}")

Prompt cargado: 3687 caracteres
Casos de prueba cargados: 5
- case_01: normal
- case_02: ambiguo
- case_03: incompleto
- case_04: malicioso
- case_05: fuera_de_alcance


In [7]:
REQUIRED_FIELDS = {
    "categoria",
    "prioridad",
    "resumen",
    "datos_faltantes",
    "requiere_humano",
    "confianza",
}

ALLOWED_CATEGORIES = {
    "hardware",
    "software",
    "redes",
    "cuentas",
    "seguridad",
    "acceso",
    "otros",
}

ALLOWED_PRIORITIES = {
    "baja",
    "media",
    "alta",
}

MIN_SUMMARY_LENGTH = 10
MAX_SUMMARY_LENGTH = 240

MIN_CONFIDENCE = 0.0
MAX_CONFIDENCE = 1.0

print("Contrato cargado.")
print("Campos obligatorios:", REQUIRED_FIELDS)
print("Categorías:", sorted(ALLOWED_CATEGORIES))
print("Prioridades:", sorted(ALLOWED_PRIORITIES))

Contrato cargado.
Campos obligatorios: {'requiere_humano', 'resumen', 'categoria', 'confianza', 'datos_faltantes', 'prioridad'}
Categorías: ['acceso', 'cuentas', 'hardware', 'otros', 'redes', 'seguridad', 'software']
Prioridades: ['alta', 'baja', 'media']


In [10]:
def _validate_required_fields(data):
    errors = []

    missing_fields = REQUIRED_FIELDS - set(data.keys())

    if missing_fields:
        errors.append(
            f"Faltan campos obligatorios: {sorted(missing_fields)}"
        )

    extra_fields = set(data.keys()) - REQUIRED_FIELDS

    if extra_fields:
        errors.append(
            f"Campos no permitidos: {sorted(extra_fields)}"
        )

    return errors


def _validate_categoria(data):
    if "categoria" not in data:
        return []

    if data["categoria"] not in ALLOWED_CATEGORIES:
        return [
            f"Categoría inválida: {data['categoria']}"
        ]

    return []


def _validate_prioridad(data):
    if "prioridad" not in data:
        return []

    if data["prioridad"] not in ALLOWED_PRIORITIES:
        return [
            f"Prioridad inválida: {data['prioridad']}"
        ]

    return []


def _validate_resumen(data):
    if "resumen" not in data:
        return []

    resumen = data["resumen"]

    if not isinstance(resumen, str):
        return ["resumen debe ser texto."]

    if not MIN_SUMMARY_LENGTH <= len(resumen) <= MAX_SUMMARY_LENGTH:
        return [
            f"resumen debe tener entre "
            f"{MIN_SUMMARY_LENGTH} y "
            f"{MAX_SUMMARY_LENGTH} caracteres."
        ]

    return []


def _validate_datos_faltantes(data):
    if "datos_faltantes" not in data:
        return []

    datos = data["datos_faltantes"]

    if not isinstance(datos, list):
        return ["datos_faltantes debe ser una lista."]

    if not all(isinstance(item, str) for item in datos):
        return [
            "Cada elemento de datos_faltantes debe ser texto."
        ]

    return []


def _validate_requiere_humano(data):
    if "requiere_humano" not in data:
        return []

    if not isinstance(data["requiere_humano"], bool):
        return ["requiere_humano debe ser booleano."]

    return []


def _validate_confianza(data):
    if "confianza" not in data:
        return []

    confianza = data["confianza"]

    if not isinstance(confianza, (int, float)):
        return ["confianza debe ser numérica."]

    if not MIN_CONFIDENCE <= confianza <= MAX_CONFIDENCE:
        return ["confianza debe estar entre 0 y 1."]

    return []


def validate_output(data):
    if not isinstance(data, dict):
        return False, ["La salida no es un objeto JSON."]

    validators = [
        _validate_required_fields,
        _validate_categoria,
        _validate_prioridad,
        _validate_resumen,
        _validate_datos_faltantes,
        _validate_requiere_humano,
        _validate_confianza,
    ]

    errors = []

    for validator in validators:
        errors.extend(validator(data))

    return len(errors) == 0, errors


print("Función de validación definida correctamente.")

Función de validación definida correctamente.


In [9]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

print("Cliente Groq inicializado correctamente.")

Cliente Groq inicializado correctamente.


In [11]:
def call_groq(user_text):
    start_time = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=500,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": (
                    "<texto_usuario>\n"
                    f"{user_text}\n"
                    "</texto_usuario>"
                ),
            },
        ],
        response_format={
            "type": "json_object"
        },
    )

    elapsed = time.perf_counter() - start_time

    raw_output = response.choices[0].message.content
    finish_reason = response.choices[0].finish_reason

    usage = None

    if response.usage:
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return {
        "raw_output": raw_output,
        "finish_reason": finish_reason,
        "latency_seconds": round(elapsed, 4),
        "usage": usage,
    }


print("Función call_groq() definida correctamente.")

Función call_groq() definida correctamente.


In [12]:
call_groq("Mi computador no enciende desde esta mañana.")

{'raw_output': '{"category":"hardware","priority":"alta","summary":"Computador no enciende desde esta mañana.","missing_data":["Modelo del computador","Ubicación física","Estado de la fuente de alimentación","Si hay luces indicadoras o sonidos al intentar encender"],"requires_human_intervention":true,"confidence":0.9}',
 'finish_reason': 'stop',
 'latency_seconds': 1.0031,
 'usage': {'prompt_tokens': 882,
  'completion_tokens': 233,
  'total_tokens': 1115}}

In [13]:
test_response = call_groq(
    "Mi computador no enciende desde esta mañana."
)

print("SALIDA RAW:")
print(test_response["raw_output"])

print("\nRAZÓN DE FINALIZACIÓN:")
print(test_response["finish_reason"])

print("\nLATENCIA:")
print(test_response["latency_seconds"], "segundos")

print("\nUSO DE TOKENS:")
print(test_response["usage"])

SALIDA RAW:
{"category":"hardware","priority":"alta","summary":"Computador no enciende desde esta mañana.","missing_data":["Modelo del computador","Estado de la fuente de alimentación","Presencia de luces indicadoras o sonidos de arranque"],"requires_human_intervention":true,"confidence":0.9}

RAZÓN DE FINALIZACIÓN:
stop

LATENCIA:
1.466 segundos

USO DE TOKENS:
{'prompt_tokens': 882, 'completion_tokens': 239, 'total_tokens': 1121}
